# Family-Ladder Scaling Induction Study -- infrastructure check

**This notebook does not run the study.** It validates the induction
infrastructure -- prompt rendering, the `noise_intens` length control and
the per-model completion budget -- against one tokenizer, on CPU, with no
AWS calls. The study itself runs headless via `run_study.py`; see
"Running" at the end of this notebook.

## Design

The quiz is fixed at `PeriodicConfig(n=9, labels=9)` and what varies is
the **model**: 7 vendor families x 3 rungs = 21 checkpoints. Holding the
quiz fixed makes an accuracy difference between rungs or families
attributable to parameter count, not a changed task.

All study config (seeds, info arms, per-model CoT toggles, prompt template,
budget derivation) lives in `run_study.py`, which this notebook imports
rather than re-declares. The roster (spec keys, analysis tags, instance
tiers) lives in `smolbench/evals/study_config.toml`; `run_study.MODELS` is
derived from it and printed below.

## Seeds and replicate count

`BASE_SEED = 0`, `n_replicates = 30`: replicate `k` runs at seed `k`, so a
results file's seed *is* its replicate index and nothing has to be
subtracted to recover it.

## Reasoning

CoT is **ON for all 21 checkpoints**. The per-vendor toggle -- which
`chat_template_kwargs` key turns thinking on -- is `run_study.COT_ARGS`,
validated against the roster at import.

## Cost warning

The study provisions up to **21 concurrent EC2 spot instances**,
`g6e.4xlarge` (1x L40S) to `p6-b200.48xlarge` (8x B200). **Nothing in this
notebook provisions anything.** Running top-to-bottom cannot launch a box.

In [ ]:
import logging
import sys
from pathlib import Path

# Import-order trap: smolbench.evals.providers.ec2 freezes its provisioning
# constants from os.environ at IMPORT time. run_study.py loads keys.env at
# module scope before its first smolbench import, so importing it FIRST --
# before any other smolbench import in this kernel -- is what keeps the
# constants correct. Do not add a smolbench import above this cell.
NB_DIR = Path.cwd() if Path.cwd().name == "induction" else Path.cwd() / "notebooks" / "induction"
assert (NB_DIR / "run_study.py").exists(), (
    f"start the kernel at the repo root or in notebooks/induction, not {Path.cwd()}"
)
logging.basicConfig(level=logging.INFO)
sys.path.insert(0, str(NB_DIR))
import run_study

In [ ]:
# spec key -> analysis tag, straight from study_config.toml via run_study.
for key, tag in run_study.MODELS.items():
    print(f"{key:32} {tag}")


In [ ]:
# Importing rather than re-declaring is what stops this notebook and a
# headless fleet run from drifting apart: a template edit or a seed change
# applied to run_study.py (the file the fleet actually executes) would
# otherwise leave this notebook silently validating stale prompts.
from run_study import (BASE_SEED, COT_ARGS, EXPERIMENT, INFO_TYPES, MODELS,
                       completion_budget, make_quizzes, template)

print(f"{len(MODELS)} models x {len(INFO_TYPES)} arms x {EXPERIMENT.n_replicates} replicates "
      f"(seeds {EXPERIMENT.seeds[0]}..{EXPERIMENT.seeds[-1]})")


## Prompt Validation


In [ ]:
# Smallest rung on the roster (tier A, single L40S) -- validating against
# ONE model's tokenizer is enough because intens/extens/zero are
# byte-identical across all 21 checkpoints; only noise_intens depends on the
# model argument (see make_quizzes' docstring in run_study.py). Downloads
# ONE tokenizer from HuggingFace and touches no AWS and no GPU.
VALIDATION_MODEL = "nemotron-3-nano-4b"
quizzes = make_quizzes(BASE_SEED, VALIDATION_MODEL)


In [ ]:
# What to check: the Context states the nine periodic rules (one line per
# label) and nothing else, followed by the single count query.
print(quizzes["intens"][0].prompt)


In [ ]:
# What to check: the Context enumerates every position of the full period
# (2,520 lines at n=9) instead of stating rules; same query as intens.
print(quizzes["extens"][0].prompt)


In [ ]:
# What to check: the Context block is EMPTY -- the query stands alone.
# This is the chance floor: a score here needed no positive information.
print(quizzes["zero"][0].prompt)


In [ ]:
from smolbench.evals.tokenization import for_model

# The invariant is PER QUESTION and EXACT -- not on average, not within a
# tolerance -- see tests/induction/test_noise_token_match.py,
# test_noise_prompt_matches_extens_token_count. noise_intens is a
# LENGTH control: if it were not exactly as long as extens in tokens, an
# intens-vs-extens accuracy gap could be explained by prompt length instead
# of information content, and the arm would be measuring nothing.
tok = for_model(VALIDATION_MODEL)
for i, (extens_q, noise_q) in enumerate(zip(quizzes["extens"], quizzes["noise_intens"])):
    n_extens, n_noise = tok.count(extens_q.prompt), tok.count(noise_q.prompt)
    assert n_noise == n_extens, (
        f"question {i}: noise_intens {n_noise:,} != extens {n_extens:,} tokens"
    )

# Print both counts so the reader can see the length control actually
# bracket the two arms it sits between.
print(f"noise_intens matched at {tok.count(quizzes['noise_intens'][0].prompt):,} tokens "
      f"(intens is {tok.count(quizzes['intens'][0].prompt):,} tokens)")



In [ ]:
# Same pre-flight arithmetic the fleet runs, per model, before provisioning
# anything -- pure CPU + a HuggingFace tokenizer fetch, no AWS.
print(completion_budget(VALIDATION_MODEL, range(BASE_SEED, BASE_SEED + EXPERIMENT.n_replicates)))


## Running

The driver runs headless from a **terminal**, at the repo root -- 21 lanes
at up to 36 hours each cannot live in one kernel session, and a dropped
kernel would orphan a billing instance:

```
.venv/bin/python notebooks/induction/run_study.py
```

Environment knobs (see `run_study.py`'s module docstring):

- `INDUCTION_MODELS` -- comma-separated spec keys; unset runs the roster.
- `INDUCTION_SHARD` -- `index/count` seed stride for one of several processes.
- `INDUCTION_STATE_FILE` -- EC2 state-file override; defaults from the tag.
- `INDUCTION_FORCE_RERUN` -- `all` or `a-b` seeds to re-collect.
- `EC2_EXPERIMENT_TAG` -- fleet-exported base tag; defaults to the study's
  `standalone_tag`.

`run_study.py --teardown` terminates the experiment's instance and is
**standalone-only**: under a fleet supervisor the lane's box is owned and
reused by the supervisor.

## Results

**Results analysis runs on remote compute, not this host.** The cell below
is gated behind an explicit opt-in so that a casual run-all on a laptop does
not pull the whole S3 results log down.

**Selection rule: earliest wins.** `sync_down()` copies S3's earliest
surviving run per (tag, info, seed) into the local layout -- see
`smolbench/evals/results_store.py`'s module docstring. A forced re-run
stays in the S3 log but is not what analysis reads.


In [ ]:
import os

ANALYSIS_HOST = os.environ.get("ANALYSIS_HOST", "").strip() == "1"
if not ANALYSIS_HOST:
    print("ANALYSIS_HOST != 1 -- skipping. Set ANALYSIS_HOST=1 on the analysis host to run.")
else:
    # sync_down() translates the append-only S3 log back into the local
    # {tag}_{info}/rep_{seed}.yaml layout. It needs THIS experiment's own
    # model -> tag map (a log key names a model, never a tag), which is why
    # it is called through EXPERIMENT rather than the module-level CLI.
    n = EXPERIMENT.sync_down()
    print(f"synced {n} objects into {EXPERIMENT.results_dir}")
    for model in MODELS:
        # summarize() reads through the results store but never provisions
        # or serves, so it cannot trigger GPU billing.
        EXPERIMENT.summarize(model)
